# Grid plotting — Spanish transmission network on an interactive map

Draw the Spanish transmission grid with **folium** and open it in the system browser.

Set `SCENARIO` in the first code cell to `2024` or one of the four 2035 scenarios. All plotting cells use that one selection.

- Pick the **2024** grid or any **2035 scenario** (`NECPEssentials`, …). The *topology*
  is identical across scenarios (same 2024 network); the scenario only selects which
  `results/<label>/` overlay is used when you colour lines by **loading** or buses by **voltage**.
- **Highlight** an arbitrary list of lines or buses (e.g. the DC-overloaded corridors from the
  reinforcement diagnostic) by passing `highlight_lines=[...]` / `highlight_buses=[...]`.

Everything goes through one function, **`plot_grid(...)`** (see the examples at the bottom).
Maps are saved to `results/grid_maps/` and opened in your browser.


In [ ]:
import webbrowser
from pathlib import Path
import pandas as pd
import numpy as np
import folium
from branca.colormap import LinearColormap

PROJECT = Path.cwd()

# ---- Select the scenario once for the whole notebook ----
# Change only this value, then Run All.
SCENARIOS = ('2024', 'GoRES', 'NECPEssentials', 'REPowerEU++', 'Trinity')
SCENARIO = '2024'
if SCENARIO not in SCENARIOS:
    raise ValueError(f'SCENARIO must be one of {SCENARIOS}; got {SCENARIO!r}')
print(f'Grid-plot scenario: {SCENARIO}')

OUT = PROJECT / 'results' / 'grid_maps'
OUT.mkdir(parents=True, exist_ok=True)

# label -> results directory.  "2024" (or None) is the base run in results/;
# any other label X writes to results/X/  (see config.toml [scenario].label).
def results_dir(scenario):
    return PROJECT / 'results' if scenario in (None, '2024') else PROJECT / 'results' / scenario

# ---- shared topology (same for every scenario) ----
# Bus_Data.csv carries a UTF-8 BOM on the first header -> utf-8-sig.
buses = pd.read_csv('Data/Bus_Data.csv', encoding='utf-8-sig')   # bus_id, voltage, y(lat), x(lon), country
BUS = buses.set_index('bus_id')
lines = pd.read_csv('Data/lines.csv')                            # line_id, bus0, bus1, voltage, ..., dc, ...
# keep only lines whose two endpoints both have coordinates
lines = lines[lines['bus0'].isin(BUS.index) & lines['bus1'].isin(BUS.index)].copy()

# colour per voltage level [kV]
VOLT_COLOR = {400: '#d62728', 320: '#9467bd', 250: '#8c564b',
              225: '#1f77b4', 220: '#1f77b4', 132: '#2ca02c'}
def volt_color(v):
    return VOLT_COLOR.get(int(v), '#7f7f7f')

print(f'{len(BUS)} buses, {len(lines)} lines  |  voltages: {sorted(lines.voltage.unique())}')

In [ ]:
# ── result overlays (optional — only needed when colouring by loading/voltage) ──
def load_line_loading(scenario, hour=None):
    """line_id -> loading_pct (peak over hours, or a single `hour`) from a scenario's
    branch_flows.csv.  Returns {} if that scenario has not been solved."""
    f = results_dir(scenario) / 'branch_flows.csv'
    if not f.exists() or f.stat().st_size == 0:
        print(f'  (no branch_flows.csv for "{scenario}") — colour_lines_by="loading" will be grey')
        return {}
    b = pd.read_csv(f)
    if hour is not None:
        b = b[b['hour'] == hour]
    return b.groupby('branch_name')['loading_pct'].max().to_dict()

def load_bus_voltage(scenario, hour=None):
    """bus_id -> mean vm_pu from a scenario's bus_voltages.csv (for colour_buses_by='voltage')."""
    f = results_dir(scenario) / 'bus_voltages.csv'
    if not f.exists() or f.stat().st_size == 0:
        return {}
    v = pd.read_csv(f)
    if hour is not None:
        v = v[v['hour'] == hour]
    return v.groupby('bus_id')['vm_pu'].mean().to_dict()

def _add_legend(m, html_items, title):
    html = (f'<div style="position:fixed; bottom:28px; left:20px; z-index:9999; background:white; '
            f'padding:8px 11px; border:1px solid #999; border-radius:5px; '
            f'font:12px/1.5 sans-serif; box-shadow:0 1px 4px rgba(0,0,0,.3);">'
            f'<b>{title}</b><br>{html_items}</div>')
    m.get_root().html.add_child(folium.Element(html))

def load_nuts3_es(cache='Data/nuts3_es.geojson'):
    """Spain NUTS-3 (province) polygons as a GeoJSON FeatureCollection.  Downloaded
    once from Eurostat GISCO and cached locally, so later calls work offline."""
    import json, urllib.request
    cache = Path(cache)
    if cache.exists():
        return json.loads(cache.read_text(encoding='utf-8'))
    url = ('https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/'
           'NUTS_RG_20M_2021_4326_LEVL_3.geojson')
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            nuts = json.load(r)
    except Exception as e:
        print(f'  (NUTS-3 download failed: {e}) — borders skipped')
        return None
    fc = {'type': 'FeatureCollection',
          'features': [f for f in nuts['features'] if f['properties']['CNTR_CODE'] == 'ES']}
    cache.write_text(json.dumps(fc), encoding='utf-8')
    print(f'  cached NUTS-3 borders -> {cache}')
    return fc

In [ ]:
def plot_grid(scenario=SCENARIO, *, color_lines_by='voltage', hour=None,
              highlight_lines=None, highlight_buses=None,
              highlight_values=None, highlight_label='severity',
              highlight_cmap=('#ffd24d', '#fc8d59', '#e34a33', '#b30000'),
              show_buses=True, show_nuts=True, bus_radius=1.6, line_weight=1.3,
              tiles='CartoDB positron', open_browser=True, filename=None):
    """Draw the Spanish transmission grid on a folium map and open it in the browser.

    scenario         : "2024" or a 2035 label ("NECPEssentials", ...). Selects the
                       results/<label>/ overlay (topology is identical across scenarios).
    color_lines_by   : 'voltage' -> colour each line by its kV level (default)
                       'loading' -> colour by peak loading_pct from that scenario's
                                    branch_flows.csv (needs a solved run)
                       'none'    -> uniform grey
    hour             : restrict a 'loading' overlay to a single delivery hour 0-23
                       (None = peak over all hours).
    highlight_lines  : list of line_id (== branch_name) to emphasise (on top of the base grid).
    highlight_buses  : list of bus_id to emphasise                    (large red markers).
    highlight_values : OPTIONAL dict {line_id: value} (e.g. reinforcement req_factor).
                       When given, highlighted lines are coloured AND thickened by value on a
                       sequential scale (deeper red = larger) with a colourbar, instead of the
                       flat magenta.  Omit it to keep the plain magenta highlight.
    highlight_label  : colourbar caption / tooltip label for highlight_values (e.g. 'req_factor').
    show_buses       : draw every bus as a small dot (toggleable in the layer control).
    show_nuts        : draw NUTS-3 (province) borders as a toggleable layer (checkbox in the map).

    Note: when highlight_values is supplied (severity colour scale), every non-highlighted
    line is drawn grey so the highlighted corridors stand out.
    """
    HL_L = set(highlight_lines or [])
    HL_B = set(highlight_buses or [])
    missing = []

    severity_mode = bool(highlight_values)   # grey out base lines so highlights stand out

    m = folium.Map(location=[40.0, -3.6], zoom_start=6, tiles=tiles, control_scale=True)

    # NUTS-3 province borders as their own toggleable layer (checkbox in the map)
    if show_nuts:
        nuts = load_nuts3_es()
        if nuts:
            fg_nuts = folium.FeatureGroup(name='NUTS-3 borders', show=True)
            folium.GeoJson(nuts, name='NUTS-3',
                           style_function=lambda _f: {'color': '#888888', 'weight': 0.8,
                                                      'fill': False, 'opacity': 0.7}).add_to(fg_nuts)
            fg_nuts.add_to(m)

    fg_lines = folium.FeatureGroup(name='transmission lines', show=True)
    fg_buses = folium.FeatureGroup(name='buses', show=show_buses)
    fg_hl_l  = folium.FeatureGroup(name='★ highlighted lines', show=True)
    fg_hl_b  = folium.FeatureGroup(name='★ highlighted buses', show=True)

    # loading overlay + colour scale
    loading, cmap = {}, None
    if color_lines_by == 'loading' and not severity_mode:
        loading = load_line_loading(scenario, hour)
        if loading:
            vmax = max(100.0, max(loading.values()))
            cmap = LinearColormap(['#1a9850', '#fee08b', '#fc8d59', '#d73027', '#7a0177'],
                                  vmin=0, vmax=vmax,
                                  caption=f'{scenario} line loading [% of rating]'
                                          + (f' — h{hour:02d}' if hour is not None else ' (peak)'))
            cmap.add_to(m)

    # base lines (skip highlighted ones — drawn on top afterwards)
    for _, r in lines.iterrows():
        lid = r['line_id']
        if lid in HL_L:
            continue
        b0, b1 = BUS.loc[r['bus0']], BUS.loc[r['bus1']]
        pts = [(b0['y'], b0['x']), (b1['y'], b1['x'])]
        kv = int(r['voltage'])
        if severity_mode:                       # grey base so highlighted corridors pop
            col, tip = '#cccccc', f'{lid} | {kv} kV'
        elif color_lines_by == 'loading':
            lp = loading.get(lid)
            col = cmap(lp) if (cmap and lp is not None) else '#cccccc'
            tip = f'{lid} | {kv} kV | loading {lp:.0f}%' if lp is not None else f'{lid} | {kv} kV | (no result)'
        elif color_lines_by == 'voltage':
            col, tip = volt_color(kv), f'{lid} | {kv} kV'
        else:
            col, tip = '#888888', f'{lid} | {kv} kV'
        dash = '5,6' if str(r['dc']).lower() in ('t', 'true') else None
        folium.PolyLine(pts, color=col, weight=line_weight, opacity=0.85,
                        dash_array=dash, tooltip=tip).add_to(fg_lines)

    # highlighted lines on top — flat magenta, or coloured/thickened by severity when
    # highlight_values ({line_id: value}, e.g. reinforcement req_factor) is supplied.
    hv = highlight_values or {}
    hcmap = None
    if hv:
        drawn_vals = [v for lid, v in hv.items()
                      if lid in HL_L and not lines[lines['line_id'] == lid].empty]
        if drawn_vals:
            lo, hi = min(drawn_vals), max(drawn_vals)
            hi = hi if hi > lo else lo + 1e-9
            hcmap = LinearColormap(list(highlight_cmap), vmin=lo, vmax=hi,
                                   caption=f'highlighted lines — {highlight_label}')
            hcmap.add_to(m)
    for lid in HL_L:
        rr = lines[lines['line_id'] == lid]
        if rr.empty:
            missing.append(lid); continue
        r = rr.iloc[0]; b0, b1 = BUS.loc[r['bus0']], BUS.loc[r['bus1']]
        val = hv.get(lid)
        if hcmap is not None and val is not None:
            col = hcmap(val)
            wt  = 2.5 + 4.0 * (val - hcmap.vmin) / (hcmap.vmax - hcmap.vmin)
            tip = f'★ {lid} | {int(r["voltage"])} kV | {highlight_label} {val:.2f}'
        else:
            col, wt, tip = '#ff00ff', 4.5, f'★ {lid} | {int(r["voltage"])} kV'
        folium.PolyLine([(b0['y'], b0['x']), (b1['y'], b1['x'])],
                        color=col, weight=wt, opacity=0.95, tooltip=tip).add_to(fg_hl_l)

    # buses
    if show_buses:
        for bid, b in BUS.iterrows():
            if bid in HL_B:
                continue
            folium.CircleMarker([b['y'], b['x']], radius=bus_radius, color='#3366cc',
                                weight=0.4, fill=True, fill_color='#3366cc', fill_opacity=0.5,
                                tooltip=f'{bid} | {int(b["voltage"])} kV').add_to(fg_buses)
    for bid in HL_B:
        if bid not in BUS.index:
            missing.append(bid); continue
        b = BUS.loc[bid]
        folium.CircleMarker([b['y'], b['x']], radius=6, color='#7a0026', weight=2,
                            fill=True, fill_color='#ff2a2a', fill_opacity=0.9,
                            tooltip=f'★ {bid} | {int(b["voltage"])} kV').add_to(fg_hl_b)

    for fg in (fg_lines, fg_buses, fg_hl_l, fg_hl_b):
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    if color_lines_by == 'voltage' and not severity_mode:
        volts = sorted(lines['voltage'].unique())
        items = '<br>'.join(f'<span style="display:inline-block;width:16px;height:3px;'
                            f'background:{volt_color(v)};vertical-align:middle;margin-right:6px;"></span>{int(v)} kV'
                            for v in volts)
        _add_legend(m, items, 'Voltage')

    if missing:
        print(f'  {len(missing)} highlight name(s) not in topology (skipped, e.g. transformers):',
              missing[:12], '...' if len(missing) > 12 else '')
    filename = filename or f'grid_{scenario}_{color_lines_by}.html'
    out = OUT / filename
    m.save(str(out))
    print('saved ->', out)
    if open_browser:
        webbrowser.open(out.resolve().as_uri())
    return m

## Maps for the selected scenario

Run whichever cell you want — each saves an HTML map to `results/grid_maps/` and opens it in the browser.


In [ ]:
# 1) The 2024 grid, coloured by voltage level (400 kV red, 220 kV blue, 132 kV green ...)
plot_grid(SCENARIO, color_lines_by='voltage')

In [ ]:
# 2) A 2035 scenario grid coloured by peak line loading from its solved run
plot_grid(SCENARIO, color_lines_by='loading')

In [ ]:
# 3) Highlight a list of LINES — here the DC-overloaded corridors from the diagnostic
diagnostic = results_dir(SCENARIO) / 'dc_overloaded_lines.csv'
if diagnostic.exists():
    ov = pd.read_csv(diagnostic)
    overloaded = ov.sort_values('req_factor', ascending=False)['branch_name'].tolist()
    plot_grid(SCENARIO, color_lines_by='voltage', highlight_lines=overloaded,
              filename=f'grid_{SCENARIO}_overloaded_lines.html')
else:
    print(f'No DC-overload diagnostic for {SCENARIO}: {diagnostic}')

In [ ]:
# 4) Highlight a list of BUSES (pass any bus_id list); lines drawn plain grey
plot_grid(SCENARIO, color_lines_by='none',
          highlight_buses=['ES00802', 'ES00123', 'ES00929', 'ES00490'],
          filename=f'grid_{SCENARIO}_highlight_buses.html')

In [ ]:
# 5) OPTIONAL severity colouring: pass highlight_values={line_id: value} to colour AND
#    thicken the highlighted lines by that value (deeper red = worse) with a colourbar,
#    instead of the flat magenta. Here: the DC reinforcement req_factor per corridor.
diagnostic = results_dir(SCENARIO) / 'dc_overloaded_lines.csv'
if diagnostic.exists():
    ov = pd.read_csv(diagnostic)
    sev = dict(zip(ov['branch_name'], ov['req_factor']))
    plot_grid(SCENARIO, color_lines_by='voltage',
              highlight_lines=list(sev), highlight_values=sev, highlight_label='req_factor',
              filename=f'grid_{SCENARIO}_overloaded_by_severity.html')
else:
    print(f'No DC-overload diagnostic for {SCENARIO}: {diagnostic}')

# tip: filter to only the worst corridors by trimming the dict first, e.g. > 1.5x:
# sev2 = {k: v for k, v in sev.items() if v > 1.5}
# plot_grid(SCENARIO, highlight_lines=list(sev2), highlight_values=sev2,
#           highlight_label='req_factor', filename='grid_overloaded_gt15.html')

In [ ]:
# 6) The line_rating_factor = 0.8 infeasibility pocket (GoRES, IIS diagnosis 2026-07-16,
#    see results/GoRES/iis.txt).  A radial pocket — ES01041/ES01042/ES01068/ES01069 plus
#    FR border bus ES00981 — whose ONLY outlet to the rest of the grid is LTGES1001
#    (ES00215–ES01041, ~491 MW nameplate -> 393 MW at the 0.8 derate).  Trapped inside:
#    ~246 MW of frozen run-of-river hydro at ES01042 and the fixed French import share
#    at ES00981 (~8.9% of net FR exchange, up to ~240 MW on 2024-12-02).  Worst hour
#    needs ~478 MW of export through the 393 MW line -> 26 hard-infeasible hours
#    (all 24 h of 2024-12-02 + 2024-07-08 h07/h23).  Lines are coloured by the worst-hour
#    flow they must carry relative to their 0.8-derated rating (>1 = physically impossible).
LRF = 0.8
ROR_MW, XB_MAX_MW, POCKET_LOAD_MW = 246.0, 240.0, 7.5   # frozen hydro | max FR share | pocket load

def nameplate_mw(lid):
    r = lines.loc[lines['line_id'] == lid].iloc[0]
    return 3**0.5 * r['voltage'] * r['Imax'] * max(r['circuits'], 1.0)

pocket_buses = ['ES00215', 'ES01041', 'ES01042', 'ES01068', 'ES01069', 'ES00981']
pocket_req = {  # worst-hour MW each line must carry / its 0.8-derated rating
    'LTGES1001':  (ROR_MW + XB_MAX_MW - POCKET_LOAD_MW) / (nameplate_mw('LTGES1001') * LRF),  # sole outlet — BINDING
    'LTGES1002':  XB_MAX_MW / (nameplate_mw('LTGES1002') * LRF),        # ES01041–ES01068 (import path)
    'LTGES1219':  XB_MAX_MW / (nameplate_mw('LTGES1219') * LRF),        # ES01069–ES00981 (import path)
    'LTGES1003a': ROR_MW / 2 / (nameplate_mw('LTGES1003a') * LRF),      # ES01041–ES01042 double circuit
    'LTGES1003b': ROR_MW / 2 / (nameplate_mw('LTGES1003b') * LRF),
}
m = plot_grid(SCENARIO, color_lines_by='none',
              highlight_lines=list(pocket_req), highlight_buses=pocket_buses,
              highlight_values=pocket_req,
              highlight_label='worst-hour flow / 0.8-derated rating',
              filename=f'grid_{SCENARIO}_infeasibility_pocket.html', open_browser=False)

# annotate the mechanism and open zoomed in on the pocket
b = BUS.loc['ES01041']
folium.Marker([b['y'], b['x']], tooltip='LRF = 0.8 infeasibility pocket',
              popup=folium.Popup(
                  '<b>Infeasibility pocket (line_rating_factor = 0.8)</b><br>'
                  'Sole outlet: <b>LTGES1001</b> — 393 MW at the 0.8 derate.<br>'
                  'Fixed injections trapped inside:<br>'
                  '&bull; 246 MW frozen run-of-river hydro @ ES01042<br>'
                  '&bull; up to 240 MW fixed FR import @ ES00981<br>'
                  'Worst hour must export ~478 MW &rArr; hard-infeasible.<br>'
                  '26 hours: all of 2024-12-02 + 2024-07-08 h07/h23.',
                  max_width=320)).add_to(m)
pts = [(BUS.loc[b_]['y'], BUS.loc[b_]['x']) for b_ in pocket_buses]
m.fit_bounds([[min(p[0] for p in pts), min(p[1] for p in pts)],
              [max(p[0] for p in pts), max(p[1] for p in pts)]], padding=(60, 60))
out = OUT / f'grid_{SCENARIO}_infeasibility_pocket.html'
m.save(str(out))
print('pocket map ->', out)
webbrowser.open(out.resolve().as_uri())